In [1]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig
from trl import SFTTrainer
import torch
import json
from peft import prepare_model_for_kbit_training

g:\AI\ai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
g:\AI\ai_env\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [3]:
MODEL_NAME = "Qwen/Qwen3-8B"


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

In [5]:
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)

In [6]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

Loading checkpoint shards: 100%|██████████| 5/5 [01:24<00:00, 16.92s/it]


In [17]:
from datasets import load_dataset
dataset = load_dataset(
    "json",
    data_files= "./dataset/V1A/V1A.jsonl"
    # data_files=  r"./dataset/V1A/newlarge.jsonl"
    

)

Generating train split: 49264 examples [00:01, 25292.48 examples/s]


In [5]:
import json
from collections import defaultdict

file_path = "./dataset/V1A/cleaned.jsonl"

field_types = defaultdict(set)

with open(file_path, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, 1):
        try:
            record = json.loads(line)

            if not isinstance(record, dict):
                print(f"Line {line_num}: Not a JSON object")
                continue

            for key, value in record.items():
                field_types[key].add(type(value).__name__)

        except json.JSONDecodeError as e:
            print(f"Invalid JSON at line {line_num}: {e}")

print("\nField types:")
for key, types in field_types.items():
    print(f"{key}: {types}")


Field types:
task: {'str'}
instruction: {'str'}
input: {'dict'}
output: {'dict'}


In [11]:
import json

file_path = "./dataset/V1A/cleaned_fixed.jsonl"

with open(file_path, "r", encoding="utf-8") as f:
    for i in range(3):
        record = json.loads(next(f))
        print(f"\nRecord {i+1}:")
        print("Input keys:", record["input"].keys())
        print("Output keys:", record["output"].keys())
        print("Input:", record["input"])
        print("Output:", record["output"])


Record 1:
Input keys: dict_keys(['conversation'])
Output keys: dict_keys(['memories'])
Input: {'conversation': 'Trainer: Ready for our marathon prep session today?\nUser: Always, though my left knee has been feeling a bit stiff lately.\nTrainer: We should definitely dial back the heavy squats then. Are you still running every morning?\nUser: Yes, I do a 5k run every morning at 6 AM.\nTrainer: Good discipline. Have you maintained your high-protein diet as well?\nUser: Yes, I strictly follow a vegetarian high-protein diet for my recovery.'}
Output: {'memories': ['User runs a 5k every morning at 6 AM', 'User follows a vegetarian high-protein diet', 'User experiences left knee stiffness during heavy squats']}

Record 2:
Input keys: dict_keys(['conversation'])
Output keys: dict_keys(['memories'])
Input: {'conversation': "Advisor: Let's review your investment goals for this financial year.\nUser: Sounds good. I want to keep my focus entirely on equity mutual funds.\nAdvisor: Any specific se

In [2]:
import json
from datasets import Dataset

input_file = "./dataset/V1A/cleaned.jsonl"

records = []

def to_text(value):
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False)

with open(input_file, "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        try:
            record = json.loads(line)

            input_data = record.get("input", {})
            output_data = record.get("output", {})

            # Normalize input
            if isinstance(input_data, dict):
                conversation = to_text(input_data.get("conversation", ""))
            else:
                conversation = to_text(input_data)

            # Normalize output
            if isinstance(output_data, dict):
                memories = output_data.get("memories", [])
            else:
                memories = output_data

            if memories is None:
                memories = []
            elif not isinstance(memories, list):
                memories = [memories]

            memories = [to_text(item) for item in memories]

            records.append({
                "task": to_text(record.get("task", "")),
                "instruction": to_text(record.get("instruction", "")),
                "input": {"conversation": conversation},
                "output": {"memories": memories}
            })

        except Exception as e:
            print(f"Problem on line {line_number}: {e}")
            print("Line content:", line[:500])
            raise

dataset = Dataset.from_list(records)
print(dataset)

Dataset({
    features: ['task', 'instruction', 'input', 'output'],
    num_rows: 49262
})


In [14]:
import json

input_file = r"./dataset/V1A/large.jsonl"
output_file = r"./dataset/V1A/newlarge.jsonl"

removed_count = 0
total_records = 0

with open(input_file, "r", encoding="utf-8") as infile, \
     open(output_file, "w", encoding="utf-8") as outfile:

    for line in infile:
        if not line.strip():
            continue

        record = json.loads(line)
        total_records += 1

        input_data = record.get("input", {})

        if isinstance(input_data, dict):
            for field in ("domain", "relationship"):
                if field in input_data:
                    del input_data[field]
                    removed_count += 1

        outfile.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Total records: {total_records}")
print(f"Fields removed: {removed_count}")
print(f"Output saved to: {output_file}")

Total records: 24594
Fields removed: 45864
Output saved to: ./dataset/V1A/newlarge.jsonl


In [ ]:
# 

In [8]:
import json

# ==========================================================
# Prepare Model
# ==========================================================

model = prepare_model_for_kbit_training(model)

model.enable_input_require_grads()
model.gradient_checkpointing_enable()
model.config.use_cache = False

# ==========================================================
# System Prompt
# ==========================================================

SYSTEM_PROMPT = """You are an expert assistant objective prediction model.

Your task is to predict the assistant's objective from the given conversation.

Rules:
- Predict only the assistant's objective.
- Do NOT generate a reply.
- Do NOT summarize the conversation.
- Do NOT extract memories.
- Do NOT infer unsupported information.
- Base your prediction only on the provided conversation.

Return ONLY valid JSON in the following format:

{
  "primary_objective": "",
  "secondary_objective": "",
  "priority": "",
  "reason": ""
}

Field Definitions:
- primary_objective: The assistant's main objective.
- secondary_objective: An optional supporting objective. Use "None" if no meaningful secondary objective exists.
- priority: One of "High", "Medium", or "Low".
- reason: A brief explanation (1–2 sentences) describing why these objectives were selected.

Return only the JSON object. Do not include any extra text or markdown.
"""

# ==========================================================
# Formatting Function
# ==========================================================

def formatting_func(example):

    user_input = (
        f"{example['instruction']}\n\n"
        f"{json.dumps(example['input'], ensure_ascii=False, separators=(',', ':'))}"
    )

    assistant_output = json.dumps(
        example["output"],
        ensure_ascii=False,
        separators=(",", ":")
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_input,
        },
        {
            "role": "assistant",
            "content": assistant_output,
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

In [9]:
# Printing  TOKEN ,DISTRIBUTION ,FORMATTING ,LONGEST ,SAMPLES of this dataset (important thing before training)

import time
from statistics import mean, median

def analyze_dataset(name, dataset, tokenizer, formatting_func):
    print("\n" + "="*70)
    print(name)
    print("="*70)

    train = dataset["train"]

    lengths = []
    format_times = []

    start_total = time.time()

    for i, sample in enumerate(train):
        t1 = time.time()

        text = formatting_func(sample)

        t2 = time.time()
        format_times.append(t2 - t1)

        tokens = tokenizer(text, add_special_tokens=True)["input_ids"]
        lengths.append(len(tokens))

        if (i + 1) % 5000 == 0:
            print(f"Processed {i+1}/{len(train)}")

    total_time = time.time() - start_total

    print("\n----- BASIC -----")
    print("Samples              :", len(train))
    print("Average Tokens       :", round(mean(lengths),2))
    print("Median Tokens        :", median(lengths))
    print("Minimum Tokens       :", min(lengths))
    print("Maximum Tokens       :", max(lengths))

    print("\n----- TOKEN DISTRIBUTION -----")
    print(">256 tokens          :", sum(x > 256 for x in lengths))
    print(">512 tokens          :", sum(x > 512 for x in lengths))
    print(">1024 tokens         :", sum(x > 1024 for x in lengths))
    print(">2048 tokens         :", sum(x > 2048 for x in lengths))
    print(">4096 tokens         :", sum(x > 4096 for x in lengths))

    print("\n----- FORMATTING -----")
    print("Formatting Time      :", round(total_time,2), "sec")
    print("Average/sample       :", round(mean(format_times)*1000,3), "ms")
    print("Samples/sec          :", round(len(train)/total_time,2))

    print("\n----- LONGEST SAMPLES -----")
    top = sorted(enumerate(lengths), key=lambda x: x[1], reverse=True)[:10]

    for idx, tok in top:
        print(f"Sample {idx:6d} : {tok} tokens")

    return lengths


In [10]:
old_dataset = load_dataset(
    "json",
    data_files= "./dataset/V1B.jsonl",
  
)

# new_dataset = load_dataset(
#     "json",
#     data_files=r"./newV4a/finalV4A.jsonl",
  
# )


old_lengths = analyze_dataset(
    "OLD DATASET",
    old_dataset,
    tokenizer,
    formatting_func
)

# new_lengths = analyze_dataset(
#     "NEW V4A DATASET",
#     new_dataset,
#     tokenizer,
#     formatting_func
# )


OLD DATASET
Processed 5000/69638
Processed 10000/69638
Processed 15000/69638
Processed 20000/69638
Processed 25000/69638
Processed 30000/69638
Processed 35000/69638
Processed 40000/69638
Processed 45000/69638
Processed 50000/69638
Processed 55000/69638
Processed 60000/69638
Processed 65000/69638

----- BASIC -----
Samples              : 69638
Average Tokens       : 445.96
Median Tokens        : 439.0
Minimum Tokens       : 263
Maximum Tokens       : 784

----- TOKEN DISTRIBUTION -----
>256 tokens          : 69638
>512 tokens          : 10564
>1024 tokens         : 0
>2048 tokens         : 0
>4096 tokens         : 0

----- FORMATTING -----
Formatting Time      : 109.26 sec
Average/sample       : 0.24 ms
Samples/sec          : 637.35

----- LONGEST SAMPLES -----
Sample  37265 : 784 tokens
Sample  22996 : 779 tokens
Sample  21287 : 777 tokens
Sample  19558 : 770 tokens
Sample  24338 : 764 tokens
Sample   5031 : 761 tokens
Sample  12116 : 759 tokens
Sample   2030 : 753 tokens
Sample  3873